In [1]:
import os
import sys
import pickle as pkl
import pandas as pd
import numpy as np
from datasets import load_dataset
from pathlib import Path
from tqdm import tqdm

/home/cyrus/venvs/uts-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Import Dataset
Dataset contains SBERT question, human response, agent response, and agent summary vectors. 

In [2]:
sys.path.append(str(Path.cwd().parent))

ROOT    = Path(os.getcwd()).resolve().parents[0]
DATA    = ROOT / "data"
WEIGHTS = ROOT / "weights"

with open(DATA / "concat_train_set.pkl", "rb") as file:
    train_set = pkl.load(file)

with open(DATA / "concat_test_set.pkl", "rb") as file:
    test_set = pkl.load(file)

train_df    = pd.DataFrame(train_set)
test_df     = pd.DataFrame(test_set)

print("Train set:", train_df.shape)
print("Test set:", train_df.shape)

Train set: (1708, 12)
Test set: (1708, 12)


## Load BERT Models

In [3]:
import torch
from transformers import (
    BertForQuestionAnswering, 
    AutoTokenizer,
)
from sentence_transformers import SentenceTransformer

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


In [5]:
bert_model = BertForQuestionAnswering.from_pretrained("deepset/bert-base-cased-squad2")
bert_model.to(device)
bert_tokenizer = AutoTokenizer.from_pretrained("deepset/bert-base-cased-squad2")
sentence_model = SentenceTransformer('paraphrase-MiniLM-L6-v2').to(device)

# Count BERT's total number of trainable parameters by summing the size of every weight tensor.
bert_num_params = sum(p.numel() for p in bert_model.parameters())
# Print the count, so "small model" isn't just a label in a markdown cell -- see the real number.
print(f"BERT ({bert_model.config.name_or_path}) parameters: {bert_num_params:,}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14531.86it/s]
[transformers] BertForQuestionAnswering LOAD REPORT from: deepset/bert-base-cased-squad2
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17099.28it/s]


BERT (deepset/bert-base-cased-squad2) parameters: 107,721,218


## Training Configuration

In [6]:
from dataclasses import dataclass

@dataclass
class Config:
    epochs: int = 50

    # Data shape (fixed by SBERT)
    question_dim: int = sentence_model.get_sentence_embedding_dimension()
    bert_dim: int     = sentence_model.get_sentence_embedding_dimension()

    # Code sizes
    style_dim: int    = 16
    content_dim: int  = 128
    context_dim: int  = 128

    # Probe outputs: filled in from the code sizes below
    style_out: int    = 1
    content_out: int  = None
    context_out: int  = None

    disc_dim: int     = sentence_model.get_sentence_embedding_dimension()
    gene_dim: int     = sentence_model.get_sentence_embedding_dimension()

    lr_generator: float     = 1e-4
    lr_discriminator: float = 1e-4
    batch_size: int         = 32

    w_align: float      = 1.0
    w_leak: float       = 1.0
    lr_probe: float     = 1e-3
    w_recon: float      = 1.0
    w_style: float      = 1.0
    w_content: float    = 1.0
    w_context: float    = 1.0
    w_retrieval: float  = 1.0

    def __post_init__(self):
        if self.content_out is None:
            self.content_out = self.content_dim
        if self.context_out is None:
            self.context_out = self.context_dim

cfg = Config(epochs=50)

/tmp/ipykernel_91306/2909750790.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  question_dim: int = sentence_model.get_sentence_embedding_dimension()
/tmp/ipykernel_91306/2909750790.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  bert_dim: int     = sentence_model.get_sentence_embedding_dimension()
/tmp/ipykernel_91306/2909750790.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  disc_dim: int     = sentence_model.get_sentence_embedding_dimension()
/tmp/ipykernel_91306/2909750790.py:22: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  gene_dim: int     = sentence_model.get_sentence_embedding_dimension()


In [7]:
from train.pairs import *

train_question  = torch.tensor(train_df["question_bert"].to_list(), dtype=torch.float32)
train_human     = torch.tensor(train_df["hr_bert"].to_list(), dtype=torch.float32)
train_agent     = torch.tensor(train_df["as_bert"].to_list(), dtype=torch.float32)

test_question   = torch.tensor(test_df["question_bert"].to_list(), dtype=torch.float32)
test_human      = torch.tensor(test_df["hr_bert"].to_list(), dtype=torch.float32)
test_agent      = torch.tensor(test_df["as_bert"].to_list(), dtype=torch.float32)

train_dataset = SummaryPairDataset(
    train_question,
    train_human,
    train_agent
)
test_dataset = SummaryPairDataset(
    test_question,
    test_human,
    test_agent
)

train_loader    = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)
test_loader     = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=True) 

/tmp/ipykernel_91306/3805931428.py:3: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  train_question  = torch.tensor(train_df["question_bert"].to_list(), dtype=torch.float32)


In [8]:
train_question.shape

torch.Size([1708, 384])

## Load Disentangler

In [9]:
from models.disentangler import SBERTDisentangler
from models.discriminator import SBERTDiscriminator
from models.generator import SBERTGenerator
from models.encoders import (
    StyleEncoder,
    ContentEncoder,
    ContextEncoder,
)
from train.step import *

In [10]:
def alter_train(model, optim_G, optim_D, optim_P, loader, n_epochs):
    history = []

    state = AlternatingState()

    for epoch in range(n_epochs):
        model.train()

        epoch_logs = {}
        n_G, n_D = 0, 0

        for batch in tqdm(loader, desc=f"epoch {epoch + 1}/{n_epochs}", leave=False):
            batch = {key: value.to(device) for key, value in batch.items()}

            logs = train_iteration(
                model=model,
                batch=batch,
                optim_G=optim_G,
                optim_D=optim_D,
                optim_P=optim_P,
                state=state,
                lambda_cross=1.0,
                lambda_leak=1.0,
                lambda_adv=1.0,
            )

            if logs["updated"] == "G":
                n_G += 1
            else:
                n_D += 1

            # Accumulate only values that exist for this update.
            for key, value in logs.items():
                if (isinstance(value, (int, float)) and np.isfinite(value)):
                    epoch_logs[key] = (epoch_logs.get(key, 0.0) + value)

        # Average G and D losses separately.
        mean_logs = {}

        for key, value in epoch_logs.items():
            if key.startswith("loss_D"):
                mean_logs[key] = value / max(n_D, 1)
            else:
                mean_logs[key] = value / max(n_G, 1)

        mean_logs["epoch"] = epoch + 1
        mean_logs["n_G"] = n_G
        mean_logs["n_D"] = n_D

        history.append(mean_logs)

        print(
            f"epoch {epoch + 1:3d} | "
            f"G updates {n_G:4d} | "
            f"D updates {n_D:4d} | "
            f"self {mean_logs.get('loss_self', float('nan')):.4f} | "
            f"cross {mean_logs.get('loss_cross', float('nan')):.4f} | "
            f"adv {mean_logs.get('loss_adv', float('nan')):.4f} | "
            f"D {mean_logs.get('loss_D', float('nan')):.4f}"
        )

    return model, history

In [11]:
def mixed_train(model, optim_G, optim_D, optim_P, loader, n_epochs):
    history = []

    for epoch in range(n_epochs):
        epoch_losses = {}

        for batch in tqdm(loader, total=len(loader), desc="Training"):
            batch = {key: value.to(device) for key, value in batch.items()}
            losses = train_step(model, batch, optim_G, optim_D, optim_P)

            for name, value in losses.items():
                epoch_losses[name] = (epoch_losses.get(name, 0.0) + value)

        for name in epoch_losses:
            epoch_losses[name] /= len(loader)
            
        history.append(epoch_losses)
        print(
            f"epoch {epoch + 1:3d} | "
            f"generator loss {epoch_losses['generator_loss']:.4f} | "
            f"self loss {epoch_losses['self_loss']:.4f} | "
            f"cross loss {epoch_losses['cross_loss']:.4f} | "
            f"adversarial loss {epoch_losses['adversarial_loss']:.4f} | "
            f"discriminator loss {epoch_losses['discriminator_loss']:.4f}"
        )

    return model, history

In [12]:
alter_model = SBERTDisentangler(cfg).to(device)
alter_optim_G = torch.optim.Adam(
    alter_model.generator_parameters(),
    lr=cfg.lr_generator
)
alter_optim_D = torch.optim.Adam(
    alter_model.discriminator_parameters(),
    lr=cfg.lr_discriminator
)
alter_optim_P = torch.optim.Adam(alter_model.probe_parameters(), lr=cfg.lr_probe)

In [13]:
alter_model, alter_hist = alter_train(
    model=alter_model, 
    optim_G=alter_optim_G, 
    optim_D=alter_optim_D, 
    optim_P=alter_optim_P, 
    loader=train_loader, 
    n_epochs=cfg.epochs * 2
)

epoch   1 | G updates   15 | D updates   39 | self 5.6097 | cross 4.2988 | adv 0.6936 | D 0.5627


epoch   2 | G updates   18 | D updates   36 | self 4.9259 | cross 4.2855 | adv 0.6991 | D 0.4112


epoch   3 | G updates   23 | D updates   31 | self 4.4737 | cross 4.2243 | adv 0.7376 | D 0.4835


epoch   4 | G updates   25 | D updates   29 | self 4.1978 | cross 4.2409 | adv 0.8743 | D 0.6111


epoch   5 | G updates   28 | D updates   26 | self 3.6878 | cross 4.0966 | adv 1.0249 | D 0.5865


...

epoch  96 | G updates   27 | D updates   27 | self 1.7383 | cross 1.3047 | adv 1.1045 | D 0.5035


epoch  97 | G updates   26 | D updates   28 | self 1.7785 | cross 1.3235 | adv 1.0219 | D 0.5833


epoch  98 | G updates   26 | D updates   28 | self 1.7550 | cross 1.2940 | adv 1.0074 | D 0.5737


epoch  99 | G updates   23 | D updates   31 | self 1.7113 | cross 1.3157 | adv 1.1298 | D 0.6455


epoch 100 | G updates   18 | D updates   36 | self 1.7004 | cross 1.3423 | adv 1.2707 | D 0.6894


In [14]:
mixed_model = SBERTDisentangler(cfg).to(device)
mixed_optim_G = torch.optim.Adam(
    mixed_model.generator_parameters(),
    lr=cfg.lr_generator
)
mixed_optim_D = torch.optim.Adam(
    mixed_model.discriminator_parameters(),
    lr=cfg.lr_discriminator
)
mixed_optim_P = torch.optim.Adam(mixed_model.probe_parameters(), lr=cfg.lr_probe)

In [15]:
mixed_model, mixed_hist = mixed_train(
    model=mixed_model, 
    optim_G=mixed_optim_G, 
    optim_D=mixed_optim_D, 
    optim_P=mixed_optim_P, 
    loader=train_loader, 
    n_epochs=cfg.epochs
)

Training: 100%|██████████| 54/54 [00:00<00:00, 55.01it/s]


epoch   1 | generator loss 12.8410 | self loss 4.9825 | cross loss 4.2652 | adversarial loss 0.6478 | discriminator loss 0.5689


Training: 100%|██████████| 54/54 [00:00<00:00, 54.66it/s]


epoch   2 | generator loss 10.7049 | self loss 3.9527 | cross loss 4.0276 | adversarial loss 0.7771 | discriminator loss 0.6175


Training: 100%|██████████| 54/54 [00:00<00:00, 54.84it/s]


...

epoch  48 | generator loss 4.7922 | self loss 1.4457 | cross loss 0.9258 | adversarial loss 0.6970 | discriminator loss 0.9825


Training: 100%|██████████| 54/54 [00:00<00:00, 58.77it/s]


epoch  49 | generator loss 4.7061 | self loss 1.4186 | cross loss 0.8425 | adversarial loss 0.6986 | discriminator loss 0.9022


Training: 100%|██████████| 54/54 [00:00<00:00, 58.31it/s]

epoch  50 | generator loss 4.8443 | self loss 1.4122 | cross loss 0.7811 | adversarial loss 0.9063 | discriminator loss 0.7382


## Evaluation

In [16]:
@torch.no_grad()
def evaluate_encoders(model, val_loader, device):
    model.eval()

    all_questions     = []
    all_human_style   = []
    all_agent_style   = []
    all_human_content = []
    all_agent_content = []
    all_human_context = []
    all_agent_context = []

    for batch in val_loader:
        question = batch["question"].to(device)
        human = batch["human"].to(device)
        agent = batch["agent"].to(device)

        # Encode the paired human and AI responses
        style_h, content_h, context_h = model.encode(human)
        style_a, content_a, context_a = model.encode(agent)

        all_questions.append(question.cpu())
        all_human_style.append(style_h.cpu())
        all_agent_style.append(style_a.cpu())
        all_human_content.append(content_h.cpu())
        all_agent_content.append(content_a.cpu())
        all_human_context.append(context_h.cpu())
        all_agent_context.append(context_a.cpu())

    # Combine batches so retrieval is evaluated across the full
    # validation set, rather than only within individual batches.
    questions = torch.cat(all_questions)
    style_h = torch.cat(all_human_style)
    style_a = torch.cat(all_agent_style)
    content_h = torch.cat(all_human_content)
    content_a = torch.cat(all_agent_content)
    context_h = torch.cat(all_human_context)
    context_a = torch.cat(all_agent_context)

    # ---------------------------------------------------------
    # 1. STYLE: human vs. AI classification accuracy
    # ---------------------------------------------------------
    style_embeddings = torch.cat([style_h, style_a], dim=0)

    style_labels = torch.cat([torch.zeros(len(style_h)),torch.ones(len(style_a)),]).long()
    style_logits = model.style_classifier(style_embeddings.to(device)).squeeze(-1).cpu()

    style_predictions = (style_logits >= 0).long()
    style_accuracy = (style_predictions == style_labels).float().mean().item()

    # ---------------------------------------------------------
    # 2. CONTENT: paired-response retrieval accuracy
    # ---------------------------------------------------------
    # Each human response should match its paired AI response.
    # Normalize vectors so matrix multiplication gives cosine
    # similarity.
    content_h = F.normalize(content_h, p=2, dim=-1)
    content_a = F.normalize(content_a, p=2, dim=-1)

    content_similarity = content_h @ content_a.T
    targets = torch.arange(len(content_h))

    # Human -> AI and AI -> human retrieval
    h_to_a = (content_similarity.argmax(dim=1) == targets).float().mean().item()
    a_to_h = (content_similarity.argmax(dim=0) == targets).float().mean().item()

    content_accuracy = (h_to_a + a_to_h) / 2

    # Also report paired cosine similarity for interpretability.
    content_paired_cosine = (content_h * content_a).sum(dim=-1).mean().item()

    # ---------------------------------------------------------
    # 3. CONTEXT: question embedding prediction
    # ---------------------------------------------------------
    # The context regressor predicts the original SBERT
    # question embedding from each context representation.
    predicted_context_h = model.context_regressor(context_h.to(device)).cpu()
    predicted_context_a = model.context_regressor(context_a.to(device)).cpu()

    predicted_context = torch.cat([predicted_context_h, predicted_context_a], dim=0)

    # Each question is a candidate target for retrieval.
    # The question embedding is shared by its human and AI pair.
    question_candidates = F.normalize(questions, p=2, dim=-1)
    predicted_context = F.normalize(predicted_context, p=2, dim=-1)

    # Repeat question indices to align with human and AI contexts.
    context_targets = torch.cat([torch.arange(len(questions)),torch.arange(len(questions)),])
    context_similarity = predicted_context @ question_candidates.T
    context_retrieval_accuracy = (context_similarity.argmax(dim=1) == context_targets).float().mean().item()

    # Cosine similarity with the correct question for each response
    true_questions = questions.repeat(2, 1)
    context_cosine = F.cosine_similarity(predicted_context, true_questions, dim=-1).mean().item()

    # ---------------------------------------------------------
    # Summary table
    # ---------------------------------------------------------
    results = pd.DataFrame([
        {
            "Encoder": "Style",
            "Metric": "Human/AI classification accuracy",
            "Score": style_accuracy,
        },
        {
            "Encoder": "Content",
            "Metric": "Paired retrieval accuracy (top-1)",
            "Score": content_accuracy,
        },
        {
            "Encoder": "Content",
            "Metric": "Paired cosine similarity",
            "Score": content_paired_cosine,
        },
        {
            "Encoder": "Context",
            "Metric": "Question retrieval accuracy (top-1)",
            "Score": context_retrieval_accuracy,
        },
        {
            "Encoder": "Context",
            "Metric": "Question cosine similarity",
            "Score": context_cosine,
        },
    ])

    results["Score"] = results["Score"].round(4)

    return results

In [17]:
device = next(alter_model.parameters()).device

results = evaluate_encoders(
    model=alter_model,
    val_loader=train_loader,
    device=device,
)

print("\nAlternating State Training Results")
print(results)

device = next(alter_model.parameters()).device

results = evaluate_encoders(
    model=alter_model,
    val_loader=test_loader,
    device=device,
)

print("\nAlternating State Testing Results")
print(results)


Alternating State Training Results
   Encoder                               Metric   Score
0    Style     Human/AI classification accuracy  0.6496
1  Content    Paired retrieval accuracy (top-1)  0.9529
2  Content             Paired cosine similarity  0.9407
3  Context  Question retrieval accuracy (top-1)  0.9031
4  Context           Question cosine similarity  0.4423

Alternating State Testing Results
   Encoder                               Metric   Score
0    Style     Human/AI classification accuracy  0.6500
1  Content    Paired retrieval accuracy (top-1)  0.9579
2  Content             Paired cosine similarity  0.9322
3  Context  Question retrieval accuracy (top-1)  0.9395
4  Context           Question cosine similarity  0.4135


In [18]:
device = next(mixed_model.parameters()).device

results = evaluate_encoders(
    model=mixed_model,
    val_loader=train_loader,
    device=device,
)

print("\nMixed State Training Results")
print(results)

device = next(mixed_model.parameters()).device

results = evaluate_encoders(
    model=mixed_model,
    val_loader=test_loader,
    device=device,
)

print("\nMixed State Testing Results")
print(results)


Mixed State Training Results
   Encoder                               Metric   Score
0    Style     Human/AI classification accuracy  0.7825
1  Content    Paired retrieval accuracy (top-1)  0.9643
2  Content             Paired cosine similarity  0.9373
3  Context  Question retrieval accuracy (top-1)  0.9297
4  Context           Question cosine similarity  0.4923

Mixed State Testing Results
   Encoder                               Metric   Score
0    Style     Human/AI classification accuracy  0.7632
1  Content    Paired retrieval accuracy (top-1)  0.9605
2  Content             Paired cosine similarity  0.9264
3  Context  Question retrieval accuracy (top-1)  0.9658
4  Context           Question cosine similarity  0.4618


## Save Models

In [20]:
alt_state   = {
    "config": cfg,
    "model": alter_model
}
mix_state   = {
    "config": cfg,
    "model": mixed_model
}

torch.save(alt_state, WEIGHTS / "alt_state.pt")
torch.save(mix_state, WEIGHTS / "mix_state.pt")